# Assignment 2 — Food-101: simple baseline vs pretrained ResNet-18

Bare-bones M2 scaffold.

| Role | Model | Where |
|---|---|---|
| Simple baseline | Assignment 1 CNN, retrained from scratch on Food-101 (128px, 101 classes) | `assignment2/models/baseline.py` |
| Pretrained/modern model | ImageNet ResNet-18, fine-tuned on Food-101 (224px, 101 classes) | `assignment2/models/resnet.py` |

- Dataset: Food-101 (75,750 train / 25,250 official test), stratified 10% validation.
- Metrics: top-1 accuracy and macro-F1.

[TODO] final hyperparameters, augmentation policy, freeze-vs-full ablation, train/val duplicate audit, curves, error analysis, multiple seeds.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "assignment2").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "assignment2"))
sys.path.insert(0, str(REPO_ROOT))

import json

import pandas as pd
import torch
import torch.nn as nn

from training_loop import evaluate, fit, set_seed
from utils.dataloader import build_food101_loaders
from models.baseline import build_baseline_cnn
from models.resnet import build_resnet18

SEED = 42
EPOCHS = 10
BATCH_SIZE = 128
WORKERS = 8
LIMIT = None  # set e.g. 1000 for a smoke run
MODELS = ["baseline", "resnet18"]  # add "resnet18_frozen" for the ablation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = REPO_ROOT / "assignment2" / "results"
CHECKPOINT_DIR = REPO_ROOT / "assignment2" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("device:", DEVICE)

device: cuda


In [2]:
loaders, class_names = build_food101_loaders(128, batch_size=8, workers=0, limit=16)
images, labels = next(iter(loaders["train"]))
print("classes:", len(class_names))
print("batch:", tuple(images.shape), images.dtype)
print("labels:", labels[:8].tolist())

classes: 101
batch: (8, 3, 128, 128) torch.float32
labels: [43, 4, 34, 7, 51, 46, 29, 60]


In [3]:
def build_model(name, num_classes, image_size):
    if name == "baseline":
        return build_baseline_cnn(num_classes=num_classes, image_size=image_size)
    if name == "resnet18":
        return build_resnet18(num_classes=num_classes, pretrained=True, freeze_backbone=False)
    if name == "resnet18_frozen":
        return build_resnet18(num_classes=num_classes, pretrained=True, freeze_backbone=True)
    raise ValueError(f"unknown model: {name}")

In [4]:
results = []
histories = {}

for name in MODELS:
    print(f"\n=== {name} ===")
    set_seed(SEED)
    image_size = 128 if name == "baseline" else 224
    loaders, class_names = build_food101_loaders(
        image_size, batch_size=BATCH_SIZE, workers=WORKERS, limit=LIMIT
    )
    model = build_model(name, len(class_names), image_size).to(DEVICE)
    frozen = name == "resnet18_frozen"
    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    checkpoint_path = CHECKPOINT_DIR / f"{name}.pt"

    history, best_epoch, train_time = fit(
        model,
        loaders,
        DEVICE,
        epochs=EPOCHS,
        lr=1e-3,
        weight_decay=1e-4,
        amp=AMP,
        checkpoint_path=checkpoint_path,
        frozen_backbone=frozen,
    )
    test = evaluate(model, loaders["test"], nn.CrossEntropyLoss(), DEVICE, AMP)

    results.append(
        {
            "model": name,
            "params": params,
            "trainable_params": trainable,
            "image_size": image_size,
            "best_epoch": best_epoch,
            "train_time_s": train_time,
            "val_accuracy": history["val_accuracy"][best_epoch - 1],
            "val_macro_f1": history["val_macro_f1"][best_epoch - 1],
            "test_accuracy": test["accuracy"],
            "test_macro_f1": test["macro_f1"],
        }
    )
    histories[name] = history
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(results)


=== baseline ===
epoch 01/10 | train loss 4.2091 acc 0.0663 | val loss 3.8797 acc 0.1100 macro-F1 0.0878 | 37.3s
epoch 02/10 | train loss 3.7268 acc 0.1391 | val loss 3.5614 acc 0.1646 macro-F1 0.1393 | 35.5s
epoch 03/10 | train loss 3.4180 acc 0.1938 | val loss 3.3553 acc 0.2038 macro-F1 0.1841 | 35.2s
epoch 04/10 | train loss 3.2091 acc 0.2320 | val loss 3.1648 acc 0.2384 macro-F1 0.2190 | 33.8s
epoch 05/10 | train loss 3.0382 acc 0.2664 | val loss 3.0910 acc 0.2532 macro-F1 0.2365 | 33.5s
epoch 06/10 | train loss 2.9050 acc 0.2928 | val loss 3.0285 acc 0.2618 macro-F1 0.2476 | 33.3s
epoch 07/10 | train loss 2.7964 acc 0.3127 | val loss 2.9664 acc 0.2785 macro-F1 0.2639 | 31.7s
epoch 08/10 | train loss 2.7044 acc 0.3314 | val loss 2.8930 acc 0.2948 macro-F1 0.2867 | 33.4s
epoch 09/10 | train loss 2.6197 acc 0.3501 | val loss 2.9318 acc 0.2978 macro-F1 0.2832 | 33.3s
epoch 10/10 | train loss 2.5548 acc 0.3621 | val loss 2.9200 acc 0.3007 macro-F1 0.2845 | 33.4s

=== resnet18 ===
epoc

,model,params,trainable_params,image_size,best_epoch,train_time_s,val_accuracy,val_macro_f1,test_accuracy,test_macro_f1
0,baseline,8829349,8829349,128,8,340.905546,0.294785,0.286701,0.330495,0.322827
1,resnet18,11228325,11228325,224,10,465.993760,0.681980,0.680573,0.722455,0.719515


In [5]:
config = {
    "dataset": "food-101",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "limit": LIMIT,
    "amp": AMP,
    "monitor": "val_macro_f1",
}
with open(RESULTS_DIR / "results.json", "w") as handle:
    json.dump({"config": config, "results": results, "history": histories}, handle, indent=2)
print("saved", RESULTS_DIR / "results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment2/results/results.json


## Next steps

[TODO] freeze-vs-full ablation, curves, error analysis, multiple seeds.